# 02 — Silver Layer: Cleaning & Transformation

**Goal:** turn the raw Bronze data into clean, analysis-ready tables.

What happens here (and only here):
- Standardize column names to snake_case (fixes the `Sp. Atk` / `Sp. Def` dot issue for every step downstream)
- Handle the known data quality issues flagged in Bronze (missing `Name`, missing `Type 2`)
- Join `combats` with Pokémon names so battles are human-readable, not just IDs
- Add a `first_pokemon_won` boolean — useful for every downstream analysis

**Input:** `data/bronze/pokemon_parquet/`, `data/bronze/combats_parquet/`
**Output:** `data/silver/pokemon_silver/`, `data/silver/combats_silver/`

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("PokemonSilverTransform")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

BRONZE_PATH = "../data/bronze"
SILVER_PATH = "../data/silver"

df_pokemon = spark.read.parquet(f"{BRONZE_PATH}/pokemon_parquet")
df_combats = spark.read.parquet(f"{BRONZE_PATH}/combats_parquet")

print(f"pokemon: {df_pokemon.count()} rows")
print(f"combats: {df_combats.count()} rows")

pokemon: 800 rows
combats: 50000 rows


## Standardize `pokemon` columns

`withColumnRenamed` matches the existing column name directly (no SQL parsing involved), so it handles names with dots/spaces without the backtick workaround we needed in Bronze.

In [2]:
df_pokemon_clean = (
    df_pokemon
    .withColumnRenamed("#", "pokemon_id")
    .withColumnRenamed("Name", "name")
    .withColumnRenamed("Type 1", "type_1")
    .withColumnRenamed("Type 2", "type_2")
    .withColumnRenamed("HP", "hp")
    .withColumnRenamed("Attack", "attack")
    .withColumnRenamed("Defense", "defense")
    .withColumnRenamed("Sp. Atk", "sp_atk")
    .withColumnRenamed("Sp. Def", "sp_def")
    .withColumnRenamed("Speed", "speed")
    .withColumnRenamed("Generation", "generation")
    .withColumnRenamed("Legendary", "legendary")
)

df_pokemon_clean.printSchema()

root
 |-- pokemon_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- type_1: string (nullable = true)
 |-- type_2: string (nullable = true)
 |-- hp: integer (nullable = true)
 |-- attack: integer (nullable = true)
 |-- defense: integer (nullable = true)
 |-- sp_atk: integer (nullable = true)
 |-- sp_def: integer (nullable = true)
 |-- speed: integer (nullable = true)
 |-- generation: integer (nullable = true)
 |-- legendary: boolean (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



## Handle known data quality issues

**`type_2` nulls:** expected and legitimate — many Pokémon only have one type. We fill with the literal string `"None"` instead of leaving it null, so it groups/filters cleanly in Power BI later (nulls are awkward in BI visuals and DAX).

**`name` nulls:** this dataset has a well-known single missing value at Pokémon `#63` (Primeape) in some CSV versions. We check for it explicitly below — if it's there, we fix it with a documented correction. We never silently guess; if the null shows up somewhere unexpected, we stop and look at it manually instead of blindly patching.

In [3]:
# Inspect any null names before touching anything
null_names = df_pokemon_clean.filter(F.col("name").isNull())
null_names.show()

# Known, documented correction for this specific public dataset
KNOWN_NAME_CORRECTIONS = {63: "Primeape"}

df_pokemon_clean = df_pokemon_clean.withColumn(
    "name",
    F.when(
        (F.col("name").isNull()) & (F.col("pokemon_id") == 63),
        F.lit(KNOWN_NAME_CORRECTIONS[63]),
    ).otherwise(F.col("name")),
)

# Confirm no unexplained nulls remain
remaining_nulls = df_pokemon_clean.filter(F.col("name").isNull()).count()
print(f"Remaining null names after correction: {remaining_nulls}")
if remaining_nulls > 0:
    print("WARNING: unexpected null names found — inspect manually before proceeding.")

+----------+----+--------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+
|pokemon_id|name|  type_1|type_2| hp|attack|defense|sp_atk|sp_def|speed|generation|legendary|_ingestion_timestamp|_source_file|
+----------+----+--------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+
|        63|NULL|Fighting|  NULL| 65|   105|     60|    60|    70|   95|         1|    false|2026-08-11 18:01:...| pokemon.csv|
+----------+----+--------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+

Remaining null names after correction: 0


In [4]:
df_pokemon_clean = df_pokemon_clean.withColumn(
    "type_2", F.coalesce(F.col("type_2"), F.lit("None"))
)

# Trim any stray whitespace on string columns (defensive — cheap to do, avoids silent join misses later)
df_pokemon_clean = (
    df_pokemon_clean
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("type_1", F.trim(F.col("type_1")))
    .withColumn("type_2", F.trim(F.col("type_2")))
)

df_pokemon_clean.show(5)

+----------+-------------+------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+
|pokemon_id|         name|type_1|type_2| hp|attack|defense|sp_atk|sp_def|speed|generation|legendary|_ingestion_timestamp|_source_file|
+----------+-------------+------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+
|         1|    Bulbasaur| Grass|Poison| 45|    49|     49|    65|    65|   45|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|         2|      Ivysaur| Grass|Poison| 60|    62|     63|    80|    80|   60|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|         3|     Venusaur| Grass|Poison| 80|    82|     83|   100|   100|   80|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|         4|Mega Venusaur| Grass|Poison| 80|   100|    123|   122|   120|   80|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|         5|   Charmander|  Fire|  None| 39|    52|    

## Write `pokemon_silver`

In [5]:
df_pokemon_silver = df_pokemon_clean.withColumn("_silver_processed_at", F.current_timestamp())

df_pokemon_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/pokemon_silver")
print("pokemon_silver written successfully.")

pokemon_silver written successfully.


## Standardize `combats` columns

In [6]:
df_combats_clean = (
    df_combats
    .withColumnRenamed("First_pokemon", "first_pokemon_id")
    .withColumnRenamed("Second_pokemon", "second_pokemon_id")
    .withColumnRenamed("Winner", "winner_id")
)

df_combats_clean.printSchema()

root
 |-- first_pokemon_id: integer (nullable = true)
 |-- second_pokemon_id: integer (nullable = true)
 |-- winner_id: integer (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



## Referential integrity check

Before joining, confirm every ID referenced in `combats` actually exists in `pokemon`. If this ever comes back non-zero, it means the join below would silently drop rows — better to know now.

In [7]:
valid_ids = df_pokemon_clean.select("pokemon_id")

orphan_first = df_combats_clean.join(
    valid_ids, df_combats_clean.first_pokemon_id == valid_ids.pokemon_id, "left_anti"
).count()

orphan_second = df_combats_clean.join(
    valid_ids, df_combats_clean.second_pokemon_id == valid_ids.pokemon_id, "left_anti"
).count()

orphan_winner = df_combats_clean.join(
    valid_ids, df_combats_clean.winner_id == valid_ids.pokemon_id, "left_anti"
).count()

print(f"Combats with invalid first_pokemon_id: {orphan_first}")
print(f"Combats with invalid second_pokemon_id: {orphan_second}")
print(f"Combats with invalid winner_id: {orphan_winner}")

Combats with invalid first_pokemon_id: 0
Combats with invalid second_pokemon_id: 0
Combats with invalid winner_id: 0


## Join combats with Pokémon names + derive `first_pokemon_won`

We join `pokemon` three times (aliased) to resolve `first_pokemon_name`, `second_pokemon_name`, and `winner_name` from IDs — this is exactly the kind of multi-join Spark handles efficiently at scale, which matters once we move to millions of simulated battles in the next notebook.

In [8]:
pokemon_names = df_pokemon_clean.select("pokemon_id", "name")

first_names = pokemon_names.withColumnRenamed("pokemon_id", "first_pokemon_id").withColumnRenamed("name", "first_pokemon_name")
second_names = pokemon_names.withColumnRenamed("pokemon_id", "second_pokemon_id").withColumnRenamed("name", "second_pokemon_name")
winner_names = pokemon_names.withColumnRenamed("pokemon_id", "winner_id").withColumnRenamed("name", "winner_name")

df_combats_silver = (
    df_combats_clean
    .join(first_names, on="first_pokemon_id", how="left")
    .join(second_names, on="second_pokemon_id", how="left")
    .join(winner_names, on="winner_id", how="left")
    .withColumn("first_pokemon_won", F.col("winner_id") == F.col("first_pokemon_id"))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

df_combats_silver.select(
    "first_pokemon_name", "second_pokemon_name", "winner_name", "first_pokemon_won"
).show(10, truncate=False)

+------------------+----------------------+----------------------+-----------------+
|first_pokemon_name|second_pokemon_name   |winner_name           |first_pokemon_won|
+------------------+----------------------+----------------------+-----------------+
|Larvitar          |Nuzleaf               |Nuzleaf               |false            |
|Virizion          |Terrakion             |Terrakion             |false            |
|Togetic           |Beheeyem              |Beheeyem              |false            |
|Slugma            |Druddigon             |Druddigon             |false            |
|Omastar           |Shuckle               |Omastar               |true             |
|Joltik            |Aegislash Shield Forme|Joltik                |true             |
|Natu              |Jynx                  |Jynx                  |false            |
|Machop            |Giratina Altered Forme|Giratina Altered Forme|false            |
|Pineco            |Clauncher             |Clauncher             

## Write `combats_silver`

In [9]:
df_combats_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/combats_silver")
print("combats_silver written successfully.")

combats_silver written successfully.


## Verify by reading back

In [10]:
check_pokemon = spark.read.parquet(f"{SILVER_PATH}/pokemon_silver")
check_combats = spark.read.parquet(f"{SILVER_PATH}/combats_silver")

print(f"pokemon_silver rows: {check_pokemon.count()}")
print(f"combats_silver rows: {check_combats.count()}")

check_pokemon.show(5)

pokemon_silver rows: 800
combats_silver rows: 50000
+----------+-------------+------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+--------------------+
|pokemon_id|         name|type_1|type_2| hp|attack|defense|sp_atk|sp_def|speed|generation|legendary|_ingestion_timestamp|_source_file|_silver_processed_at|
+----------+-------------+------+------+---+------+-------+------+------+-----+----------+---------+--------------------+------------+--------------------+
|         1|    Bulbasaur| Grass|Poison| 45|    49|     49|    65|    65|   45|         1|    false|2026-08-11 18:01:...| pokemon.csv|2026-08-12 18:11:...|
|         2|      Ivysaur| Grass|Poison| 60|    62|     63|    80|    80|   60|         1|    false|2026-08-11 18:01:...| pokemon.csv|2026-08-12 18:11:...|
|         3|     Venusaur| Grass|Poison| 80|    82|     83|   100|   100|   80|         1|    false|2026-08-11 18:01:...| pokemon.csv|2026-08-12 18:11:...|
|         4|

In [11]:
spark.stop()